# 05 vLLM 之外的 Serving 框架：如何做技术选型

vLLM 很重要，但 AI Infra 不等于 vLLM。不同框架服务不同目标：通用吞吐、结构化生成、多节点编排、NVIDIA 极致优化、本地低成本、生态兼容。产品/研发要掌握的是选型逻辑，而不是背框架名字。


## 1. Serving 框架解决什么问题

一个 LLM serving 框架通常要处理：

- 模型加载：权重格式、量化、分片、lazy loading。
- 请求接口：OpenAI-compatible API、streaming、工具调用、structured output。
- 调度：continuous batching、prefill/decode、优先级、队列。
- 显存管理：KV cache、paged attention、prefix caching、eviction。
- 并行：tensor/pipeline/data/expert parallel。
- 监控：metrics、logs、traces、错误码。
- 部署：Docker、Kubernetes、多节点、autoscaling。

```mermaid
flowchart TB
    API["API 兼容性 / API compatibility"] --> Engine["推理服务引擎 / Serving engine"]
    Engine --> Schedule["调度与批处理 / scheduler / batching"]
    Engine --> Memory["KV 缓存与内存管理 / KV cache / memory manager"]
    Engine --> Parallel["并行策略 / parallelism"]
    Engine --> Metrics["指标与追踪 / metrics / tracing"]
    Engine --> Deploy["部署集成 / deployment integration"]
```


## 2. 框架定位对比

| 框架 | 核心定位 | 强项 | 注意点 |
|---|---|---|---|
| vLLM | 通用高吞吐 LLM serving | PagedAttention、prefix cache、OpenAI API、生态成熟 | 调优依赖 workload，Mac 需 vLLM-Metal |
| SGLang | 高性能 serving + 结构化/程序化生成 | RadixAttention、prefix caching、structured outputs、多 GPU | 需要理解 runtime 和 grammar backend |
| TensorRT-LLM | NVIDIA GPU 极致优化栈 | engine/runtime、in-flight batching、paged attention、quantization | NVIDIA 绑定强，构建和部署复杂 |
| TGI | Hugging Face 生态 serving | router、continuous batching、SSE、Prometheus/OTel | 官方已进入 maintenance mode，应谨慎作为新项目主选 |
| Ray Serve LLM | 分布式 LLM serving 编排 | 多节点、多模型、autoscaling、PD disaggregation、routing | 更像平台编排层，可配合 vLLM/SGLang |
| llama.cpp/MLX | 本地/边缘/低成本推理 | CPU/Mac/量化、本地开发友好 | 生产高并发能力和生态不同 |


## 3. Ray Serve LLM 的位置

Ray Serve LLM 不是单纯替代 vLLM 的 engine，而是偏“分布式 serving 编排”。官方强调它支持 OpenAI-compatible API、多节点多模型、autoscaling、metrics/Grafana、PD disaggregation，以及可组合 vLLM/SGLang 等 engine。

这类框架适合：

- 多模型、多租户、多节点服务。
- 需要自定义 routing，例如 prefix-aware 或 session-aware routing。
- 需要把模型服务和 Python 业务逻辑组合在一个分布式系统里。
- 需要自动扩缩容和故障恢复。

如果你的团队只是单机跑一个 7B 模型，Ray Serve 可能显得重；如果你要做平台级多模型服务，它的价值会更明显。


## 4. SGLang 与结构化生成

SGLang 的重要特点是面向低延迟、高吞吐 inference，并强调 RadixAttention、prefix caching、多 GPU parallelism 和 structured outputs。Structured outputs 允许用 JSON schema、regex 或 EBNF 约束输出，让模型输出满足结构要求。

这对 Agent/RAG 很重要：很多工具调用、表单抽取、风控分类、SQL/DSL 生成都不能只靠“模型大概会按格式输出”。约束解码能把格式错误从“事后解析失败”提前到“生成过程中约束”。

代价是：结构约束会引入 grammar backend、schema 设计和性能权衡。复杂 schema 可能降低生成速度或导致模型难以满足约束。


## 5. TensorRT-LLM 与 NVIDIA 优化栈

TensorRT-LLM 适合在 NVIDIA GPU 上追求极致性能。它通常涉及 engine 构建、runtime 执行、量化、in-flight batching、paged attention、Triton integration 等。你可以把它理解成：比通用 Python serving 更贴近 NVIDIA 推理优化栈。

适合场景：

- 大规模生产，硬件以 NVIDIA 为主。
- 模型和 workload 相对稳定，值得投入 engine 构建和性能调优。
- 对吞吐/延迟/成本非常敏感。

风险：部署复杂度更高，模型更新和兼容性验证成本更高，团队需要更强系统和 GPU 工程能力。


## 6. TGI 的现实定位

Hugging Face TGI 曾经推动了很多开源 LLM serving 能力：router、continuous batching、SSE streaming、Prometheus/OTel、tensor parallelism、quantization、Flash/Paged Attention 等。现在官方文档明确说明 TGI 进入 maintenance mode，并推荐 vLLM、SGLang 以及 llama.cpp/MLX 等下游引擎。

这不代表 TGI 没价值。它仍然是理解 serving 架构的好材料，也可能在已有系统中继续运行。但如果你从零做新平台，应把维护状态纳入选型风险。


In [ ]:
from dataclasses import dataclass

@dataclass
class Workload:
    name: str
    local_mac: bool = False
    structured: bool = False
    nvidia_max_perf: bool = False
    distributed_platform: bool = False
    generic_serving: bool = True


def recommend(w: Workload):
    if w.local_mac:
        return 'MLX / llama.cpp / vLLM-Metal'
    if w.distributed_platform:
        return 'Ray Serve LLM + vLLM/SGLang engine'
    if w.nvidia_max_perf:
        return 'TensorRT-LLM'
    if w.structured:
        return 'SGLang'
    return 'vLLM'

cases = [
    Workload('Mac 本地学习', local_mac=True),
    Workload('结构化工具调用服务', structured=True),
    Workload('NVIDIA 大规模固定模型', nvidia_max_perf=True),
    Workload('多模型平台', distributed_platform=True),
    Workload('通用企业 RAG API'),
]
for c in cases:
    print(f'{c.name:18s} -> {recommend(c)}')


## 7. 选型决策流

```mermaid
flowchart TB
    Need["需求"] --> Local{"必须本地/Mac?"}
    Local -->|是| MLX["本地后端 / MLX / llama.cpp / vLLM-Metal"]
    Local -->|否| Platform{"平台级多节点多模型?"}
    Platform -->|是| Ray["平台编排 / Ray Serve LLM + engine"]
    Platform -->|否| NVIDIA{"NVIDIA 极致性能?"}
    NVIDIA -->|是| TRT["NVIDIA 优化栈 / TensorRT-LLM"]
    NVIDIA -->|否| Structured{"强结构化输出?"}
    Structured -->|是| SGLang["结构化生成 / SGLang"]
    Structured -->|否| VLLM["vLLM 默认候选"]
```

好的选型报告应该包含：workload、模型、硬件、SLA、成本、团队能力、迁移风险、benchmark 结果，而不是只写“某框架性能最好”。


## 8. 选型案例：企业知识库助手

需求：企业内部 RAG，主要是文本问答，要求私有化部署，峰值中等，prompt 较长，有共享 system prompt 和工具说明。

合理路径：

1. 先用 vLLM 建 baseline，因为通用、生态成熟、OpenAI-compatible API 易接入。
2. 开启/评估 prefix caching，因为 system prompt 和工具说明共享度高。
3. 如果结构化工具调用格式错误多，再评估 SGLang structured outputs。
4. 如果规模变成多部门多模型平台，再考虑 Ray Serve LLM 或更完整的网关编排。
5. 如果固定 NVIDIA 大规模生产且成本压力极高，再评估 TensorRT-LLM 投入是否值得。

这个案例说明：选型是阶段性的，不是一开始就选最复杂的栈。


## 9. 选型案例：本地开发助手

需求：开发者 Mac 本地跑小模型，目标是隐私、低成本、快速实验，不追求生产高并发。

合理路径：

- 优先 MLX、llama.cpp 或 vLLM-Metal。
- 模型选 1.5B/3B/7B 量化版本，根据内存和速度折中。
- 不要用本地结果外推生产吞吐。
- 保持 OpenAI-compatible client 抽象，未来切到 vLLM/SGLang/云模型更容易。


In [ ]:
# 简单选型打分：实际项目应把权重换成团队真实优先级。
frameworks = {
    'vLLM': {'maturity': 5, 'structured': 3, 'nvidia_perf': 4, 'local_mac': 2, 'ops_complexity': 3},
    'SGLang': {'maturity': 4, 'structured': 5, 'nvidia_perf': 4, 'local_mac': 1, 'ops_complexity': 4},
    'TensorRT-LLM': {'maturity': 4, 'structured': 2, 'nvidia_perf': 5, 'local_mac': 0, 'ops_complexity': 5},
    'MLX/llama.cpp': {'maturity': 4, 'structured': 2, 'nvidia_perf': 1, 'local_mac': 5, 'ops_complexity': 2},
}
weights = {'maturity': 2, 'structured': 1, 'nvidia_perf': 1, 'local_mac': 0, 'ops_complexity': -1}
for name, scores in frameworks.items():
    total = sum(scores[k] * w for k, w in weights.items())
    print(f'{name:14s} score={total}')


## 10. 迁移成本：选型时最容易低估的部分

Serving 框架不是只有启动命令不同。迁移可能影响：API 兼容性、chat template、tool calling、structured output、采样参数、metrics 名称、错误码、模型权重格式、量化格式、Kubernetes 部署、dashboard、压测脚本和回滚策略。

因此选型时要问：如果三个月后从 vLLM 换到 SGLang，业务代码要改多少？如果从云模型 fallback 到本地模型，输出格式是否一致？如果从 TGI 迁移到 vLLM，Prometheus 指标和告警如何迁移？这些问题比“hello world 能不能跑”更接近真实成本。


## 11. 选型报告模板

一份有用的 serving 选型报告应包含：

1. Workload：prompt/output token 分布、QPS、并发、SLA。
2. 模型：参数量、上下文、量化、chat template、工具/结构化能力。
3. 硬件：GPU/Metal/CPU、显存、带宽、多卡互联。
4. 候选框架：vLLM、SGLang、TensorRT-LLM、Ray Serve LLM、MLX/llama.cpp 等。
5. Benchmark：TTFT、ITL、throughput、OOM、启动时间、资源占用。
6. 运维：Docker/K8s、metrics、日志、trace、升级、回滚。
7. 风险：维护状态、社区活跃度、团队熟悉度、迁移成本。
8. 结论：当前阶段推荐方案和未来触发重新选型的条件。


## 12. 自测题

1. Ray Serve LLM 和 vLLM 的定位有什么不同？
2. 为什么 structured outputs 会影响 serving 框架选型？
3. TensorRT-LLM 适合什么团队和场景？
4. TGI 进入 maintenance mode 对新项目意味着什么？
5. 为什么保持 OpenAI-compatible client 抽象能降低迁移成本？


## 13. 从框架能力到组织能力

很多团队选型时只比较技术指标，却忽略组织能力。TensorRT-LLM 可能性能很好，但如果团队没有 NVIDIA 优化栈经验，模型经常变，部署和验证成本可能超过收益。Ray Serve LLM 适合平台化，但如果团队没有多节点服务和 Ray 运维经验，早期可能过重。SGLang 的 structured output 很有吸引力，但如果业务只是普通 chat/RAG，先用 vLLM 建 baseline 可能更快。

选型的成熟做法是分阶段：先用最小复杂度跑通可观测 baseline，再根据明确瓶颈升级框架。比如先用 vLLM；当格式约束成为主要失败来源时，引入 SGLang；当多模型多节点成为平台瓶颈时，引入 Ray Serve；当单 token 成本成为核心矛盾且硬件固定为 NVIDIA 时，再评估 TensorRT-LLM。这样每次复杂度增加都有明确收益。


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
